# PyTorch基礎

In [29]:
import torch

## torch.Tensor
GPUでの計算や勾配情報保持などの機能を持つ配列のデータ型

In [30]:
tensor = torch.tensor([1, 2, 3])
print(tensor)

tensor([1, 2, 3])


基本的な演算子はNumPyと同じ

In [31]:
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])

print(a + b) # 要素ごとの和

c = torch.tensor([[1], [2], [3]])
d = a + c
print(d) # ブロードキャストにより、cがaの形状に合わせて自動的に拡張されてから和が計算される
print(d.dtype) # データ型はtorch.int64になる


tensor([5, 7, 9])
tensor([[2, 3, 4],
        [3, 4, 5],
        [4, 5, 6]])
torch.int64


## Tensorの型
Tensorの作り方で型が変わる

In [32]:
a = torch.Tensor([1, 2, 3])
print(a.dtype) # torch.float32

b = torch.LongTensor([1, 2, 3])
print(b.dtype) # torch.int64

c = torch.FloatTensor([1, 2, 3])
print(c.dtype) # torch.float32

# torch.tensorでは与えた要素の型でTensorの型が決まる
d = torch.tensor([1, 2, 3])
print(d.dtype) # torch.int64

e = torch.tensor([1.0, 2.0, 3.0])
print(e.dtype) # torch.float32



torch.float32
torch.int64
torch.float32
torch.int64
torch.float32


## 基本的な関数

In [33]:
a = torch.tensor([1, 2, 3])
print(torch.sum(a))

tensor(6)


### バッチ処理

In [34]:
batch_size = 10

a = torch.randn(batch_size, 3, 2) # 10個の3x2行列を含むTensor
print(a.shape) # torch.Size([10, 3, 2])
print(a[0]) # 最初の3x2行列を表示

b = torch.randn(batch_size, 2, 4) # 10個の2x4行列を含むTensor
print(b.shape) # torch.Size([10, 2, 4])
print(b[0]) # 最初の2x4行列を表示

# 行列積のバッチ処理
c = torch.bmm(a, b) # バッチ行列積
print(c.shape) # torch.Size([10, 3, 4])
print(c[0]) # 最初の3x4行列を表示

torch.Size([10, 3, 2])
tensor([[-0.4223,  0.7199],
        [-1.3414,  1.6541],
        [-0.2528, -0.3700]])
torch.Size([10, 2, 4])
tensor([[ 1.5400, -0.5036,  0.7768, -0.6347],
        [-0.3149, -1.4685, -1.1064, -0.0279]])
torch.Size([10, 3, 4])
tensor([[-0.8771, -0.8446, -1.1245,  0.2480],
        [-2.5867, -1.7535, -2.8720,  0.8053],
        [-0.2728,  0.6707,  0.2130,  0.1708]])


## torch.nn
深層学習モデルの構成要素が定義されたモジュール

In [35]:
import torch.nn as nn

### nn.Embeddings
埋め込み層を作成するクラス  
インデックスとベクトルのデーブルを作成する

In [36]:
embedding = nn.Embedding(10, 3) # 埋め込みベクトルのテーブル(10個の3次元ベクトル)
print(embedding.weight) # 埋め込みベクトルのテーブルを表示

indices = torch.LongTensor([1, 3, 5]) # インデックスのTensorを作成(インデックス1,3,5を指定)

print(embedding(indices)) # embeddingからインデックス1,3,5に対応する埋め込みベクトルを取得して表示

Parameter containing:
tensor([[ 0.5510, -0.4425, -1.5098],
        [-0.5120, -0.1730,  0.2488],
        [ 0.0225, -0.4544,  1.1738],
        [-0.1031,  0.7632,  0.4467],
        [ 1.2197,  1.3702, -0.8780],
        [ 0.4457, -0.6863,  1.1358],
        [-1.4643,  0.4149,  1.4093],
        [ 0.7599, -0.9102,  0.7604],
        [ 0.2542,  0.7030, -0.4842],
        [-0.5386, -1.7642, -0.5958]], requires_grad=True)
tensor([[-0.5120, -0.1730,  0.2488],
        [-0.1031,  0.7632,  0.4467],
        [ 0.4457, -0.6863,  1.1358]], grad_fn=<EmbeddingBackward0>)


### nn.Linear
線形層(全結合層)を作成するクラス

In [37]:
input_dim = 128
output_dim = 64

linear = nn.Linear(input_dim, output_dim) # 線形層を作成

input_vector = torch.randn(input_dim) # 128次元の入力ベクトルを作成
print(input_vector.shape) # torch.Size([128])

batch_size = 10
input_batch = torch.randn(batch_size, input_dim) # 10個の128次元
print(linear(input_vector).shape) # torch.Size([64]) 線形層に入力ベクトルを通すと64次元の出力が得られる

torch.Size([128])
torch.Size([64])


### nn.Module
nn.Embeddingsやnn.Linearなどのモジュールを使って任意のレイヤーを作成するためのクラス。  
モデルを定義する際に継承して使用する。

In [38]:
# カスタムモデルのクラスを定義
class CustomModel(nn.Module):
    def __init__(self): # モデルの初期化(コンストラクタ)
        super(CustomModel, self).__init__() # 親クラスnn.Moduleのコンストラクタを呼び出す
        self.linear1 = nn.Linear(128, 64) # 線形層1を定義
        self.linear2 = nn.Linear(64, 10) # 線形層2を定義

    def forward(self, x): # 順伝播の定義
        x = self.linear1(x) # 線形層1を通す
        x = self.linear2(x) # 線形層2を通す
        return x

In [39]:
# モデルの利用
model = CustomModel() # カスタムモデルのインスタンスを作成
input_dim = 128
input_vector = torch.randn(input_dim) # 128次元の入力ベクトルを作成
output = model(input_vector) # モデルに入力ベクトルを通す

print(output) # モデルの出力を表示
print(output.shape) # torch.Size([10]) モデルの出力は10次元

tensor([ 0.2440,  0.1722,  0.0183,  0.2837,  0.6085,  0.2059, -0.1837,  0.6656,
        -0.0087,  0.6063], grad_fn=<ViewBackward0>)
torch.Size([10])


In [40]:
# モデル構成の確認
print(model) # モデルの構成を表示

CustomModel(
  (linear1): Linear(in_features=128, out_features=64, bias=True)
  (linear2): Linear(in_features=64, out_features=10, bias=True)
)


### nn.ModuleList
nn.Modelを継承したクラスのリストを作成するクラス。  
複数のレイヤーをリストで管理するために使用する。    
nn.ModuleListを使用すると、リスト内のレイヤーがモデルの一部として認識され、パラメータの更新などが正しく行われる。

(注意)よくニューラルネットワークの定義でモジュールを順次実行するのに用いられる`nn.Sequential`とは意味が異なるので注意。

In [41]:
class CustomModel(nn.Module):
    def __init__(self):
        super(CustomModel, self).__init__()
        units = [128, 64, 32, 16, 10]
        self.layers = nn.ModuleList(
            [nn.Linear(units[i], units[i+1]) for i in range(len(units)-1)]
        ) # モジュールリストを作成

    def forward(self, x):
        for layer in self.layers: # モジュールリスト内の各レイヤーを順番に通す
            x = layer(x)
        return x

4つの全結合層を持つモデルとして認識される。

In [42]:
model = CustomModel() # カスタムモデルのインスタンスを作成
print(model) # モデルの構成を表示

CustomModel(
  (layers): ModuleList(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): Linear(in_features=64, out_features=32, bias=True)
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): Linear(in_features=16, out_features=10, bias=True)
  )
)


(参考) 全結合ニューラルネットワークの例  
`nn.Sequential`を使用して、全結合層と活性化関数を順番に定義している。

```py
class MNISTModel(nn.Module):
    """MNIST用のシンプルな全結合ニューラルネットワーク"""
    
    def __init__(self, hidden_size_1=512, hidden_size_2=256):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(28 * 28, hidden_size_1), # 全結合層
            nn.ReLU(), # 活性化関数(ReLU)
            nn.Linear(hidden_size_1, hidden_size_2), # 全結合層
            nn.ReLU(), # 活性化関数(ReLU)
            nn.Linear(hidden_size_2, 10), # 出力層(最後はsoftmaxで確率計算のために10次元に変換)
            nn.LogSoftmax(dim=1), # 出力を確率に変換するためのLogSoftmax関数
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(-1, 28 * 28)
        return self.model(x)
```

### nn.functional
nnモジュールの関数型APIを提供するモジュール。

In [43]:
import torch.nn.functional as F # 関数型APIをインポート

In [44]:
x = torch.randn(10) # 10次元の入力ベクトルを作成

# nn直下のクラスを使うパターン
relu_func = nn.ReLU() # ReLU関数のインスタンスを作成
y1 = relu_func(x) # ReLU関数をインスタンスを通して適用
print(y1) # ReLU関数を適用した結果を表示

# nn.functionalを使うパターン
y2 = F.relu(x) # 関数型APIを直接呼び出してReLU関数を適用
print(y2) # ReLU関数を適用した結果を表示

tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.1311, 0.9061, 0.0000, 0.3728,
        0.0000])
tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.1311, 0.9061, 0.0000, 0.3728,
        0.0000])


## 学習に必要なモジュール

### 損失関数

In [45]:
# クロスエントロピー損失関数の例
criterion = nn.CrossEntropyLoss() # クロスエントロピー損失関数

# モデルの予測値ダミーデータ
# 3クラスの予測値(確率分布)を2サンプル用意
predicted = torch.tensor([[0.1, 0.2, 0.7], [0.8, 0.1, 0.1]], requires_grad=True) 
# 教師データ
target = torch.tensor([2, 0]) # 教師データとしてクラス番号を与える
# target = torch.tensor([[0, 0, 1], [1, 0, 0]]) # 教師データをone-hotベクトルで与える場合

loss = criterion(predicted, target) # 損失を計算
print(loss) # 計算された損失を表示

tensor(0.7288, grad_fn=<NllLossBackward0>)


### オプティマイザ
損失関数を最小化するためにパラメータ更新を行う。

In [46]:
# AdamWの例
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001) # AdamWオプティマイザを作成
print(optimizer)

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.01
)


### スケジューラ
オプティマイザに設定された学習率をエポック1ごとに調整する。

In [47]:
# StepLRの例
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1) # StepLRスケジューラを作成

## データセット、データローダー

### Dataset
データセット全体を表現するためのクラス。

In [48]:
import torch
from torch.utils.data import Dataset

class CustomDataset(Dataset): # Datasetクラスを継承してカスタムデータセットクラスを定義
    def __init__(self, data, targets):
        self.data = data # データを保存
        self.targets = targets # ターゲットを保存

    def __len__(self): # データセットのサンプル数(サイズ)を返す(len(dataset)のように呼び出す)
        return len(self.data)

    def __getitem__(self, idx): # インデックスのサンプル数を返す(dataset[0]のように呼び出す)
        return self.data[idx], self.targets[idx]

In [49]:
# ダミーデータの作成
data = torch.randn(100, 3, 32, 32) # 100個の3x32x32の画像データを作成(3はチャンネル数、32x32は画像の高さと幅)
targets = torch.randint(0, 10, (100,)) # 100個のクラス番号(0~9)をランダムに生成してターゲットとする
print(targets[:10]) # 最初の10個のターゲットを表示

dataset = CustomDataset(data, targets) # カスタムデータセットのインスタンスを作成
print(len(dataset)) # データセットのサイズを表示

# データセットから最初のサンプルを取得して表示
sample_data, sample_target = dataset[0] # データセットの最初のサンプルを取得
print(sample_data.shape) # データの形状を表示(3x32x32)
print(sample_target) # ターゲットを表示(クラス番号)

tensor([8, 4, 0, 0, 1, 4, 5, 6, 1, 0])
100
torch.Size([3, 32, 32])
tensor(8)


### DataLoader
Datasetからバッチ単位(ミニバッチ)でデータを取り出すためのクラス。

In [50]:
from torch.utils.data import DataLoader

dataloader = DataLoader(dataset, batch_size=4, shuffle=True) # データローダーを作成

for data, targets in dataloader: # データローダーからバッチ単位でデータを取り出す
    print(data.shape) # データの形状を表示 (torch.Size([4, 3, 32, 32]))
    print(targets.shape) # ターゲットの形状を表示 (torch.Size([4]))
    break # 最初のバッチだけ表示してループを抜ける

torch.Size([4, 3, 32, 32])
torch.Size([4])


## 学習の基本的な流れ

In [51]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


# カスタムデータセットクラスを定義
class CustomDataset(Dataset): 
    def __init__(self, data, targets):
        self.data = data # データを保存
        self.targets = targets # ターゲットを保存

    def __len__(self): # データセットのサンプル数(サイズ)を返す(len(dataset)のように呼び出す)
        return len(self.data)

    def __getitem__(self, idx): # インデックスのサンプル数を返す(dataset[0]のように呼び出す)
        return self.data[idx], self.targets[idx]

# カスタムモデルのクラスを定義
class CustomModel(nn.Module):
    def __init__(self):
        super(CustomModel, self).__init__()
        units = [1024, 512, 256, 128, 64, 32, 16, 10]
        self.layers = nn.ModuleList(
            [nn.Linear(units[i], units[i+1]) for i in range(len(units)-1)]
        ) # モジュールリストを作成


    def forward(self, x):
        for layer in self.layers: # モジュールリスト内の各レイヤーを順番に通す
            x = layer(x)
        return x

# ダミーデータの作成
data = torch.randn(100, 32, 32) # 100個の32x32の画像データを作成(32x32は画像の高さと幅)
# ダミーデータをモデルの入力に合わせて変形
data = data.view(100, -1) # 100個の(32*32)次元のベクトルに変形
targets = torch.randint(0, 10, (100,)) # 100個のクラス番号(0~9)をランダムに生成してターゲットとする

# Datasetの作成
dataset = CustomDataset(data, targets) # カスタムデータセットのインスタンスを作成
# DataLoaderの作成
dataloader = DataLoader(dataset, batch_size=4, shuffle=True) # データローダーを作成

n_epochs = 100
model = CustomModel() # カスタムモデルのインスタンスを作成
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001) # AdamWオプティマイザを作成
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1) # StepLRスケジューラを作成
criterion = nn.CrossEntropyLoss() # クロスエントロピー損失関数(生のlogitsを受け取って内部でlog_softmaxにかける)

# 学習ループ
for epoch in range(n_epochs):
    for data, targets in dataloader: # データローダーからバッチ単位でデータを取り出す
        optimizer.zero_grad() # 勾配を初期化
        output = model(data) # モデル出力の計算
        loss = criterion(output, targets) # 損失関数を計算
        loss.backward() # 勾配を計算
        optimizer.step() # オプティマイザでパラメータを更新
    # evaluate(model, eval_data) # 検証データに関して損失や制度を計算
    scheduler.step() # エポックごとにスケジューラで学習率を更新

In [52]:
# 適当なデータで予測値とターゲットを比較
predicted = model(data[0].unsqueeze(0)) # モデルに最後のミニバッチの先頭のサンプルを入力して予測値を計算(logitsの形で出力される)
probs = torch.softmax(predicted, dim=1) # 予測値を確率分布に変換
print(probs) # 予測値の確率分布を表示
print(probs.sum(dim=1))  # 1.0

target = targets[0] # 最後のミニバッチの先頭のサンプルのターゲットを取得

print(target) # ターゲットを表示

tensor([[4.0186e-08, 1.8488e-05, 5.2573e-07, 9.9985e-01, 1.9221e-07, 9.5044e-05,
         3.8530e-05, 2.3388e-09, 6.5863e-10, 8.2672e-07]],
       grad_fn=<SoftmaxBackward0>)
tensor([1.], grad_fn=<SumBackward1>)
tensor(3)
